Cell 1 — Fresh Clone, Exact SHA Check, Hardware Detection & Gate 1 (154 / 11 / 136)

In [12]:
from pathlib import Path
import pandas as pd

REPO_DIR = Path("/content/HFG")
RUN_ID = "run_20261111_002"

%cd /content/HFG

!python -m src.ingestion \
    --raw-csv data/raw/survey_raw.csv \
    --run-id run_20261111_002

p = REPO_DIR / "data/normalized/answers_normalized.parquet"

assert p.exists(), f"Missing {p}"

df = pd.read_parquet(p)

print("Rows:", len(df))
print("Participants:", df["participant_id"].nunique())
print(
    "Observed usable:",
    int((df["missingness_state"] == "observed_usable").sum())
)

assert len(df) == 154
assert df["participant_id"].nunique() == 11
assert (df["missingness_state"] == "observed_usable").sum() == 136

print("NORMALIZED_DATA = PASS")

/content/HFG
Ingesting raw survey from data/raw/survey_raw.csv...
Successfully normalized 154 answer records across 11 participants.
Epistemic Status Breakdown:
  substantive: 131
  raw_blank: 10
  mixed_uncertainty_and_preference: 5
  ambiguous: 2
  explicit_no_opinion: 2
  explicit_insufficient_experience: 2
  punctuation_uninterpretable: 2
Missingness State Breakdown:
  observed_usable: 136
  missing_raw_blank: 10
  missing_non_substantive: 6
  ambiguous: 2
Text supplied: 144 / 154
Has usable preference: 136 / 154
Rows: 154
Participants: 11
Observed usable: 136
NORMALIZED_DATA = PASS


In [ ]:
Cell 2 — Mount Google Drive & Configure Durable HFG_CHECKPOINT_ROOT

In [13]:
from src.hardcoding_scanner import verify_gate_1
import json

gate1 = verify_gate_1(
    base_dir=REPO_DIR,
    run_id=RUN_ID,
)

print(json.dumps(gate1, indent=2))

assert gate1["status"] == "PASS"

print("GATE_1 = PASS")

{
  "gate": "Gate 1 (Control Plane, Hardcoding Scanner, Ingestion & Question Registry)",
  "run_id": "run_20261111_002",
  "status": "PASS",
  "timestamp_utc": "2026-11-14T10:33:33.993419+00:00",
  "checks": {
    "hardcoding_scan": {
      "status": "PASS",
      "violations_count": 0,
      "violations": []
    },
    "raw_source_integrity": {
      "status": "PASS",
      "expected_sha256": "56acded3e02be49be27d6b288fa2c43fcce68df24525113eba1b1161eec52a95",
      "actual_sha256": "56acded3e02be49be27d6b288fa2c43fcce68df24525113eba1b1161eec52a95"
    },
    "question_registry": {
      "status": "PASS",
      "substantive_questions_count": 13,
      "expected_substantive_count": 13,
      "total_questions_count": 14,
      "covered_q_ids": [
        "Q01",
        "Q02",
        "Q03",
        "Q04",
        "Q05",
        "Q06",
        "Q07",
        "Q08",
        "Q09",
        "Q10",
        "Q11",
        "Q12",
        "Q13"
      ]
    },
    "missingness_preservation": {
   

In [14]:
from google.colab import drive
import os
from pathlib import Path

drive.mount("/content/drive", force_remount=False)
CHECKPOINT_ROOT = Path("/content/drive/MyDrive/HFG/run_20261111_002/checkpoints")
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HFG_CHECKPOINT_ROOT"] = str(CHECKPOINT_ROOT)

probe = CHECKPOINT_ROOT / ".write_read_probe.tmp"
probe.write_text("HFG_DURABLE_DRIVE_OK\n", encoding="utf-8")
assert probe.read_text(encoding="utf-8").strip() == "HFG_DURABLE_DRIVE_OK"
probe.unlink()
print("CHECKPOINT_ROOT =", CHECKPOINT_ROOT.resolve())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CHECKPOINT_ROOT = /content/drive/MyDrive/HFG/run_20261111_002/checkpoints


Cell 3 Light

In [15]:
REPO_DIR = Path("/content/HFG")
CHECKPOINT_ROOT = "/content/drive/MyDrive/HFG/run_20261111_002/checkpoints"

In [17]:
import json
import os
import subprocess
import sys
from pathlib import Path
from google.colab import drive

REPO_DIR = Path("/content/HFG")
RUN_ID = "run_20261111_002"

# Keep the SAME persistent Google Drive root you already validated.
CHECKPOINT_ROOT = (
    "/content/drive/MyDrive/HFG/run_20261111_002/checkpoints"
)

# 1. Ensure Drive is mounted
drive.mount("/content/drive", force_remount=False)

# 2. Update repository to canonical merged main
subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "origin", "--prune"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(REPO_DIR), "checkout", "main"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"],
    check=True,
)

head = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

print("CURRENT_MAIN_SHA =", head)

# 3. Verify that the PR #9 implementation commit is in main
SMOKE_FIX_SHA = "3b00fa76a59f0c544aaa6754d31ed97633a05e5d"

subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "merge-base", "--is-ancestor",
        SMOKE_FIX_SHA, "HEAD",
    ],
    check=True,
)

print("SMOKE_FIX_IN_MAIN = True")

# 4. Configure execution
os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.environ["HFG_EXECUTION_PROFILE"] = "smoke_tiny"
os.environ["HFG_CHECKPOINT_ROOT"] = CHECKPOINT_ROOT

print("CHECKPOINT_ROOT =", CHECKPOINT_ROOT)

# 5. Kill orphan servers from previous failed run
subprocess.run(
    "pkill -f 'vllm.entrypoints.openai.api_server' || true",
    shell=True,
)
subprocess.run(
    "pkill -f 'EngineCore' || true",
    shell=True,
)

# 6. IMPORTANT: restart-safe imports from merged main
from src.ingest.ingestion import ACTIVE_RUN_ID
from src.runtime import run_dual_coder_production
from src.pipeline.orchestrator import PipelineOrchestrator

print("ACTIVE_RUN_ID =", ACTIVE_RUN_ID)

# 7. Execute ONLY corrected smoke_tiny
smoke_result = run_dual_coder_production(
    base_dir=REPO_DIR,
    run_id=ACTIVE_RUN_ID,
    execution_profile="smoke_tiny",
    checkpoint_root=CHECKPOINT_ROOT,
    require_live_gpu=True,
    startup_timeout_sec=900.0,
)

print("\n=== SMOKE_TINY RESULT ===")
print(json.dumps(smoke_result, indent=2))

# 8. Agent 03 MUST remain blocked after smoke
orch_report = PipelineOrchestrator(
    base_dir=REPO_DIR,
    run_id=ACTIVE_RUN_ID,
).evaluate(write_log=True)

print(
    "\nAGENT_03_RELEASE =",
    orch_report.agent_03_release_decision.value,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CURRENT_MAIN_SHA = 69feb14b6ee836e67b7b8d20f0dedbe3262fe7d3
SMOKE_FIX_IN_MAIN = True
CHECKPOINT_ROOT = /content/drive/MyDrive/HFG/run_20261111_002/checkpoints


ImportError: cannot import name 'ExecutionProfile' from 'src.runtime.model_runtime' (/content/HFG/src/runtime/model_runtime.py)

In [20]:
import json
import os
import subprocess
import sys
from pathlib import Path
from google.colab import drive

REPO_DIR = Path("/content/HFG")
RUN_ID = "run_20261111_002"

# Keep the SAME persistent Google Drive root you already validated.
CHECKPOINT_ROOT = (
    "/content/drive/MyDrive/HFG/run_20261111_002/checkpoints"
)

# 1. Ensure Drive is mounted
drive.mount("/content/drive", force_remount=False)

# 2. Update repository to canonical merged main
subprocess.run(
    ["git", "-C", str(REPO_DIR), "fetch", "origin", "--prune"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(REPO_DIR), "checkout", "main"],
    check=True,
)

subprocess.run(
    ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"],
    check=True,
)

head = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

print("CURRENT_MAIN_SHA =", head)

# 3. Verify that the PR #9 implementation commit is in main
SMOKE_FIX_SHA = "3b00fa76a59f0c544aaa6754d31ed97633a05e5d"

subprocess.run(
    [
        "git", "-C", str(REPO_DIR),
        "merge-base", "--is-ancestor",
        SMOKE_FIX_SHA, "HEAD",
    ],
    check=True,
)

print("SMOKE_FIX_IN_MAIN = True")

# 4. Configure execution
os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.environ["HFG_EXECUTION_PROFILE"] = "smoke_tiny"
os.environ["HFG_CHECKPOINT_ROOT"] = CHECKPOINT_ROOT

print("CHECKPOINT_ROOT =", CHECKPOINT_ROOT)

# 5. Kill orphan servers from previous failed run
subprocess.run(
    "pkill -f 'vllm.entrypoints.openai.api_server' || true",
    shell=True,
)
subprocess.run(
    "pkill -f 'EngineCore' || true",
    shell=True,
)

# 6. IMPORTANT: restart-safe imports from merged main
from src.ingest.ingestion import ACTIVE_RUN_ID
from src.runtime import run_dual_coder_production
from src.pipeline.orchestrator import PipelineOrchestrator

print("ACTIVE_RUN_ID =", ACTIVE_RUN_ID)

# 7. Execute ONLY corrected smoke_tiny
smoke_result = run_dual_coder_production(
    base_dir=REPO_DIR,
    run_id=ACTIVE_RUN_ID,
    execution_profile="smoke_tiny",
    checkpoint_root=CHECKPOINT_ROOT,
    require_live_gpu=True,
    startup_timeout_sec=900.0,
)

print("\n=== SMOKE_TINY RESULT ===")
print(json.dumps(smoke_result, indent=2))

# 8. Agent 03 MUST remain blocked after smoke
orch_report = PipelineOrchestrator(
    base_dir=REPO_DIR,
    run_id=ACTIVE_RUN_ID,
).evaluate(write_log=True)

print(
    "\nAGENT_03_RELEASE =",
    orch_report.agent_03_release_decision.value,
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CURRENT_MAIN_SHA = 69feb14b6ee836e67b7b8d20f0dedbe3262fe7d3
SMOKE_FIX_IN_MAIN = True
CHECKPOINT_ROOT = /content/drive/MyDrive/HFG/run_20261111_002/checkpoints


ImportError: cannot import name 'ExecutionProfile' from 'src.runtime.model_runtime' (/content/HFG/src/runtime/model_runtime.py)

In [ ]:
Cell 3 — Step 3 & Step 4: Run smoke_tiny + Verify Clean Shutdown + Verify Durable Checkpoint Persistence

In [3]:
import json, subprocess
from pathlib import Path
from src.runtime import AtomicQuestionCheckpointStore, run_dual_coder_production
from src.pipeline.orchestrator import Agent03ReleaseDecision, PipelineOrchestrator

smoke_res = run_dual_coder_production(
    base_dir=Path("/content/HFG"),
    run_id="run_20261111_002",
    execution_profile="smoke_tiny",
    checkpoint_root=os.environ["HFG_CHECKPOINT_ROOT"],
    require_live_gpu=True,
    startup_timeout_sec=600.0,
)
print("SMOKE_TINY_SUMMARY =", json.dumps(smoke_res, indent=2))
assert smoke_res["status"] == "PASS"
assert smoke_res["execution_profile"] == "smoke_tiny"
assert smoke_res["production_eligible"] is False
assert smoke_res["synthetic_or_test_execution"] is True
assert all(smoke_res["smoke_diagnostics"].values())

# Check zero orphan vLLM / EngineCore processes
ps_check = subprocess.run(["pgrep", "-af", "vllm|EngineCore"], capture_output=True, text=True)
print("ORPHAN_VLLM_CHECK =", ps_check.stdout.strip() or "NONE")
assert not ps_check.stdout.strip(), f"Orphan vLLM processes found:\n{ps_check.stdout}"

# Negative Gate Check: smoke_tiny must keep Agent 03 BLOCKED
orch_after_smoke = PipelineOrchestrator(base_dir=Path("/content/HFG"), run_id="run_20261111_002").evaluate(write_log=False)
print("AGENT_03_AFTER_SMOKE =", orch_after_smoke.agent_03_release_decision.value)
assert orch_after_smoke.agent_03_release_decision == Agent03ReleaseDecision.BLOCKED

# Step 4: Durable Google Drive checkpoint proof (without deleting records)
smoke_ns = Path(os.environ["HFG_CHECKPOINT_ROOT"]) / "run_20261111_002" / "smoke_tiny"
drive_files = sorted(str(p) for p in smoke_ns.rglob("*") if p.is_file())
print("DRIVE_SMOKE_CHECKPOINT_FILES_COUNT =", len(drive_files))
print("\n".join(drive_files[:15]))
assert any("/content/drive/" in f for f in drive_files)

reloaded_smoke = AtomicQuestionCheckpointStore(
    checkpoint_dir=os.environ["HFG_CHECKPOINT_ROOT"],
    run_id="run_20261111_002",
    execution_profile="smoke_tiny",
)
recs = reloaded_smoke.all_records()
print("RELOADED_SMOKE_RECORDS =", len(recs), "QUESTIONS =", reloaded_smoke.list_completed_questions())
assert len(recs) > 0
sample = recs[0]
hit = reloaded_smoke.get(
    run_id=sample.run_id,
    question_id=sample.question_id,
    coder_id=sample.coder_id,
    prompt_hash=sample.prompt_hash,
    source_hash=sample.source_hash,
    participant_id=sample.participant_id,
    execution_profile="smoke_tiny",
    model_repo=sample.model_repo,
    model_revision=sample.model_revision,
    quantization=sample.quantization,
    raise_on_mismatch=True,
)
assert hit is not None
# Verify production namespace remains completely empty
prod_probe = AtomicQuestionCheckpointStore(os.environ["HFG_CHECKPOINT_ROOT"], run_id="run_20261111_002", execution_profile="production")
assert len(prod_probe.all_records()) == 0, "Smoke contaminated production checkpoint namespace!"
print("DURABLE_CHECKPOINT_VALIDATION = PASS")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


DualCoderExecutionError: coder_b failed on (PARTICIPANT_002, Q06, RESP_002_Q06) [error_class=LengthExhaustion, finish_reason=length, attempts=3, repair_attempted=True]: Model-based repair failed validation: JSON parse error: Failed to parse or mechanically repair JSON from LLM response without altering meaning: Expecting ',' delimiter: line 51 column 6 (char 3164)
Raw content snippet: {
  "participant_id": "PARTICIPANT_002",
  "question_id": "Q06",
  "response_id": "RESP_002_Q06",
  "is_substantive": true,
  "missingness_state": null,
  "response_status": "valid",
  "detected_language": "French",
  "evidence_spans": [
    {
      "verbatim_quote": "Les pôles devraient avant tout 

In [ ]:
Cell 4 — Steps 5, 6, 7 & 8: Run pilot_subset + Inspect coder_b / PARTICIPANT_002 / Q06 + Negative Gate + PILOT_DECISION

In [10]:
import json
import pandas as pd
from pathlib import Path
from src.runtime import (
    AtomicQuestionCheckpointStore,
    PINNED_PRODUCTION_CODER_A_REPO,
    PINNED_PRODUCTION_CODER_A_REVISION,
    PINNED_PRODUCTION_CODER_B_REPO,
    PINNED_PRODUCTION_CODER_B_REVISION,
    run_dual_coder_production,
    select_profile_dataframe,
)
from src.pipeline.orchestrator import Agent03ReleaseDecision, PipelineOrchestrator

norm_df = pd.read_parquet("/content/HFG/data/normalized/answers_normalized.parquet")
pilot_df = select_profile_dataframe(norm_df, "pilot_subset")
print("PILOT_ROWS =", len(pilot_df), "STATUSES =", sorted(pilot_df["epistemic_status"].astype(str).unique()))

pilot_res = run_dual_coder_production(
    base_dir=Path("/content/HFG"),
    run_id="run_20261111_002",
    execution_profile="pilot_subset",
    checkpoint_root=os.environ["HFG_CHECKPOINT_ROOT"],
    require_live_gpu=True,
    startup_timeout_sec=900.0,
)
print("PILOT_SUBSET_SUMMARY =", json.dumps(pilot_res, indent=2))
assert pilot_res["coder_a_metadata"]["model_repo"] == PINNED_PRODUCTION_CODER_A_REPO
assert pilot_res["coder_a_metadata"]["model_revision"] == PINNED_PRODUCTION_CODER_A_REVISION
assert pilot_res["coder_b_metadata"]["model_repo"] == PINNED_PRODUCTION_CODER_B_REPO
assert pilot_res["coder_b_metadata"]["model_revision"] == PINNED_PRODUCTION_CODER_B_REVISION
assert pilot_res["production_eligible"] is False
assert pilot_res["synthetic_or_test_execution"] is False

# Deep inspection for coder_b / PARTICIPANT_002 / Q06
pilot_store = AtomicQuestionCheckpointStore(
    checkpoint_dir=os.environ["HFG_CHECKPOINT_ROOT"],
    run_id="run_20261111_002",
    execution_profile="pilot_subset",
)
p002_q06_recs = [
    r for r in pilot_store.all_records()
    if r.coder_id == "coder_b" and r.question_id == "Q06" and r.participant_id == "PARTICIPANT_002"
]
diag_path = Path(os.environ["HFG_CHECKPOINT_ROOT"]) / "run_20261111_002" / "pilot_subset" / "failure_diagnostics.jsonl"
p002_diags = []
if diag_path.is_file():
    for line in diag_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            obj = json.loads(line)
            if obj.get("coder_id") == "coder_b" and obj.get("question_id") == "Q06" and obj.get("participant_id") == "PARTICIPANT_002":
                p002_diags.append(obj)

payload_b = json.loads(Path(pilot_res["coder_b_output_path"]).read_text(encoding="utf-8"))
p002_q06_codings = [c for c in payload_b["coding_results"] if c["participant_id"] == "PARTICIPANT_002" and c["question_id"] == "Q06"]
print("P002_Q06_DIAGNOSTICS =", json.dumps(p002_diags, indent=2))
print("P002_Q06_CHECKPOINT =", json.dumps([r.to_dict() for r in p002_q06_recs], indent=2))
print("P002_Q06_FINAL_CODING =", json.dumps(p002_q06_codings, indent=2))
print("UNRESOLVED_FAILURES =", pilot_store.list_failed_items())

# Step 7: Pilot must keep Agent 03 BLOCKED
orch_after_pilot = PipelineOrchestrator(base_dir=Path("/content/HFG"), run_id="run_20261111_002").evaluate(write_log=False)
print("AGENT_03_AFTER_PILOT =", orch_after_pilot.agent_03_release_decision.value)
assert orch_after_pilot.agent_03_release_decision == Agent03ReleaseDecision.BLOCKED

pilot_ok = (
    smoke_res["status"] == "PASS"
    and pilot_res["status"] == "PASS"
    and all(pilot_res["validation_checks"].values())
    and len(p002_q06_codings) >= 1
    and len(pilot_store.list_failed_items()) == 0
)
print("PILOT_DECISION =", "PASS_TO_PRODUCTION" if pilot_ok else "STOP")

PILOT_ROWS = 10 STATUSES = ['ambiguous', 'explicit_insufficient_experience', 'explicit_no_opinion', 'raw_blank', 'substantive']
PILOT_SUBSET_SUMMARY = {
  "status": "PASS",
  "run_id": "run_20261111_002",
  "execution_profile": "pilot_subset",
  "production_eligible": false,
  "synthetic_or_test_execution": false,
  "profile_config_hash": "",
  "archived_smoke_checkpoint_path": null,
  "execution_mode_final": "SEQUENTIAL_FALLBACK",
  "fallback_events": [],
  "rows_selected": 10,
  "questions_selected": [
    "Q02",
    "Q06",
    "Q08",
    "Q10",
    "Q11"
  ],
  "coder_a_output_path": "/content/HFG/data/processed/profiles/pilot_subset/coder_a_outputs.json",
  "coder_b_output_path": "/content/HFG/data/processed/profiles/pilot_subset/coder_b_outputs.json",
  "coder_a_metadata": {
    "coder_id": "coder_a",
    "execution_profile": "pilot_subset",
    "production_eligible": false,
    "synthetic_or_test_execution": false,
    "profile_config_hash": "",
    "model_family": "deepseek",
  

NameError: name 'smoke_res' is not defined

In [18]:
# Recompute the pilot decision from the results already in memory.

pilot_ok = (
    smoke_result["status"] == "PASS"
    and pilot_res["status"] == "PASS"
    and all(pilot_res["validation_checks"].values())
    and len(p002_q06_codings) >= 1
    and len(unresolved) == 0
)

PILOT_DECISION = (
    "PASS_TO_PRODUCTION"
    if pilot_ok
    else "STOP"
)

print("SMOKE_TINY =", smoke_result["status"])
print("PILOT_SUBSET =", pilot_res["status"])
print(
    "P002_Q06_FINAL_CODING_COUNT =",
    len(p002_q06_codings),
)
print(
    "UNRESOLVED_FAILURES =",
    len(unresolved),
)
print(
    "A-H VALIDATION =",
    all(pilot_res["validation_checks"].values()),
)

print("\n============================")
print("PILOT_DECISION =", PILOT_DECISION)
print("============================")

NameError: name 'smoke_result' is not defined

In [ ]:
Cell 5 — Steps 9–14: 5-Stage Live Benchmark (Fresh VM Precondition) -> Full production (SEQUENTIAL_FALLBACK) -> Agent 11 Validation

(Run ONLY if Cell 4 printed PILOT_DECISION = PASS_TO_PRODUCTION.)

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/HFG")
RUN_ID = "run_20261111_002"

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git", "clone",
        "--branch", "main",
        "https://github.com/necantis/HFG.git",
        str(REPO_DIR),
    ],
    check=True,
)

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

HEAD_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    text=True,
).strip()

print("CURRENT_MAIN_SHA =", HEAD_SHA)

# Ensure all production fixes are contained in current main.
required_commits = [
    "15586839f0722be19a9626f9386cdb6d0ce40c78",  # JSON recovery
    "345fd701bf309d5d007502c0a951fd2cf901d35a",  # execution profiles
    "3b00fa76a59f0c544aaa6754d31ed97633a05e5d",  # smoke correction
]

for commit in required_commits:
    subprocess.run(
        ["git", "merge-base", "--is-ancestor", commit, "HEAD"],
        check=True,
    )

print("REQUIRED_RUNTIME_FIXES_IN_MAIN = True")

# Dependencies
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "vllm>=0.7.0",
        "huggingface_hub>=0.25.0",
        "pydantic>=2.0.0",
        "pandas>=2.0.0",
        "pyarrow>=14.0.0",
        "pyyaml>=6.0.0",
    ],
    check=True,
)


CURRENT_MAIN_SHA = 85ff51cffae1276d52feeab7fe1f716543de72b8
REQUIRED_RUNTIME_FIXES_IN_MAIN = True


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'vllm>=0.7.0', 'huggingface_hub>=0.25.0', 'pydantic>=2.0.0', 'pandas>=2.0.0', 'pyarrow>=14.0.0', 'pyyaml>=6.0.0'], returncode=0)

In [2]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive", force_remount=False)

CHECKPOINT_ROOT = Path(
    "/content/drive/MyDrive/HFG/run_20261111_002/checkpoints"
)

assert CHECKPOINT_ROOT.exists(), (
    f"Persistent checkpoint root missing: {CHECKPOINT_ROOT}"
)

os.environ["HFG_CHECKPOINT_ROOT"] = str(CHECKPOINT_ROOT)
os.environ["HFG_EXECUTION_PROFILE"] = "production"

prod_cp = (
    CHECKPOINT_ROOT
    / RUN_ID
    / "production"
)

print("CHECKPOINT_ROOT =", CHECKPOINT_ROOT)
print("PRODUCTION_NAMESPACE =", prod_cp)
print("Production namespace exists:", prod_cp.exists())

if prod_cp.exists():
    files = [
        p for p in prod_cp.rglob("*")
        if p.is_file()
    ]

    print("Production checkpoint files:", len(files))

    for p in sorted(files)[:50]:
        print(
            p.relative_to(CHECKPOINT_ROOT),
            p.stat().st_size,
        )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CHECKPOINT_ROOT = /content/drive/MyDrive/HFG/run_20261111_002/checkpoints
PRODUCTION_NAMESPACE = /content/drive/MyDrive/HFG/run_20261111_002/checkpoints/run_20261111_002/production
Production namespace exists: True
Production checkpoint files: 306
run_20261111_002/production/bindings/coder_a.json 339
run_20261111_002/production/bindings/coder_b.json 358
run_20261111_002/production/checkpoint_wal.jsonl 2327511
run_20261111_002/production/failure_diagnostics.jsonl 8402
run_20261111_002/production/records/coder_a/run_20261111_002__production__Q01__CORPUS_DISCOV_3df5300f7aba15b2cab39c77.json 4617
run_20261111_002/production/records/coder_a/run_20261111_002__production__Q01__PARTICIPANT_0_17c07fd0d4b82ee8d68fe540.json 14970
run_20261111_002/production/records/coder_a/run_20261111_002__production__Q01__PARTICIPANT_0_26e37aacb69ff07c109084a4.json 13673
run_20261111_

In [3]:
!nvidia-smi

Sat Nov 14 10:39:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   28C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
%cd /content/HFG

!python -m src.ingestion \
    --raw-csv data/raw/survey_raw.csv \
    --run-id run_20261111_002

/content
Ingesting raw survey from data/raw/survey_raw.csv...
Successfully normalized 154 answer records across 11 participants.
Epistemic Status Breakdown:
  substantive: 131
  raw_blank: 10
  mixed_uncertainty_and_preference: 5
  ambiguous: 2
  explicit_no_opinion: 2
  explicit_insufficient_experience: 2
  punctuation_uninterpretable: 2
Missingness State Breakdown:
  observed_usable: 136
  missing_raw_blank: 10
  missing_non_substantive: 6
  ambiguous: 2
Text supplied: 144 / 154
Has usable preference: 136 / 154


In [5]:
import pandas as pd
from src.hardcoding_scanner import verify_gate_1

p = REPO_DIR / "data/normalized/answers_normalized.parquet"

assert p.exists()

df = pd.read_parquet(p)

n_rows = len(df)
n_participants = df["participant_id"].nunique()
n_usable = int(
    (df["missingness_state"] == "observed_usable").sum()
)

print("Rows =", n_rows)
print("Participants =", n_participants)
print("Observed usable =", n_usable)

assert n_rows == 154
assert n_participants == 11
assert n_usable == 136

gate1 = verify_gate_1(
    base_dir=REPO_DIR,
    run_id=RUN_ID,
)

print("Gate 1 =", gate1["status"])

assert gate1["status"] == "PASS"

Rows = 154
Participants = 11
Observed usable = 136
Gate 1 = PASS


In [6]:
BENCHMARK = (
    REPO_DIR
    / "runs"
    / RUN_ID
    / "gpu_runtime_benchmark.json"
)

print("Benchmark exists:", BENCHMARK.exists())

Benchmark exists: False


If Benchmark is False we need to run this cell (It takes 5 minutes)

In [7]:
from src.runtime import build_and_save_runtime_config

cfg, benchmark, cfg_path = build_and_save_runtime_config(
    base_dir=REPO_DIR,
    run_id=RUN_ID,
    resolve_remote_models=True,
    verify_tokenizer=False,
    startup_timeout_sec=900.0,
)

print(
    "live_gpu_benchmark_status =",
    cfg.live_gpu_benchmark_status,
)

print("Benchmark:", BENCHMARK)
print("Benchmark exists:", BENCHMARK.exists())

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


live_gpu_benchmark_status = EXECUTED_LIVE_GPU
Benchmark: /content/HFG/runs/run_20261111_002/gpu_runtime_benchmark.json
Benchmark exists: True


In [8]:
from src.runtime import AtomicQuestionCheckpointStore

prod_store = AtomicQuestionCheckpointStore(
    checkpoint_dir=str(CHECKPOINT_ROOT),
    run_id=RUN_ID,
    execution_profile="production",
)

records = prod_store.all_records()

print("Existing production checkpoint records:", len(records))
print(
    "Existing unresolved failures:",
    prod_store.list_failed_items(),
)

from collections import Counter

print(
    "Records by coder:",
    Counter(r.coder_id for r in records),
)

print(
    "Questions represented:",
    sorted(set(r.question_id for r in records)),
)

Existing production checkpoint records: 300
Existing unresolved failures: []
Records by coder: Counter({'coder_a': 150, 'coder_b': 150})
Questions represented: ['Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12', 'Q13', 'Q14']


In [9]:
import hashlib, json
from pathlib import Path
from src.runtime import (
    AtomicQuestionCheckpointStore,
    ExecutionMode,
    build_and_save_runtime_config,
    run_dual_coder_production,
)
from src.pipeline.orchestrator import PipelineOrchestrator

REPO_DIR = Path("/content/HFG")
RUN_ID = "run_20261111_002"
bench_path = REPO_DIR / "runs" / RUN_ID / "gpu_runtime_benchmark.json"

# Run the 5-stage live GPU benchmark in this fresh Colab VM so runs/<run_id>/gpu_runtime_benchmark.json exists,
# then enforce SEQUENTIAL_FALLBACK as the empirically safe production policy.
cfg, bench_report, cfg_path = build_and_save_runtime_config(
    base_dir=REPO_DIR,
    run_id=RUN_ID,
    resolve_remote_models=True,
    verify_tokenizer=False,
    startup_timeout_sec=900.0,
)
cfg.execution_mode = ExecutionMode.SEQUENTIAL_FALLBACK
cfg.derived_from_benchmark = False
cfg.save(cfg_path)
print("PRODUCTION_CHECKPOINT_NS =", Path(os.environ["HFG_CHECKPOINT_ROOT"]) / RUN_ID / "production")

prod_res = run_dual_coder_production(
    base_dir=REPO_DIR,
    run_id=RUN_ID,
    execution_profile="production",
    checkpoint_root=os.environ["HFG_CHECKPOINT_ROOT"],
    require_live_gpu=True,
    startup_timeout_sec=900.0,
    resolve_remote_models=False,
)
for label, p_str in (("CODER_A_SHA256", prod_res["coder_a_output_path"]), ("CODER_B_SHA256", prod_res["coder_b_output_path"])):
    print(label, "=", hashlib.sha256(Path(p_str).read_bytes()).hexdigest())
prod_store = AtomicQuestionCheckpointStore(os.environ["HFG_CHECKPOINT_ROOT"], run_id=RUN_ID, execution_profile="production")
print("PRODUCTION_UNRESOLVED_FAILURES =", prod_store.list_failed_items())
print("PRODUCTION_SUMMARY =", json.dumps(prod_res, indent=2))

# Step 13: Agent 11 PipelineOrchestrator release check (authoritative)
orch = PipelineOrchestrator(base_dir=REPO_DIR, run_id=RUN_ID)
final_report = orch.evaluate(write_log=True)
print(json.dumps(final_report.to_dict(), indent=2))
print(f"AGENT_03_RELEASE = {final_report.agent_03_release_decision.value}")

PRODUCTION_CHECKPOINT_NS = /content/drive/MyDrive/HFG/run_20261111_002/checkpoints/run_20261111_002/production
CODER_A_SHA256 = 8939adc884672a1cdf883e496f476a872248d90d4f8da8f3549d29e24b20dadf
CODER_B_SHA256 = 129c7257c1c4fdf9f3b88067b96180cf68ebd3056f41e0880fd5831d48ffcd49
PRODUCTION_UNRESOLVED_FAILURES = []
PRODUCTION_SUMMARY = {
  "status": "PASS",
  "run_id": "run_20261111_002",
  "execution_profile": "production",
  "production_eligible": true,
  "synthetic_or_test_execution": false,
  "profile_config_hash": "",
  "archived_smoke_checkpoint_path": null,
  "execution_mode_final": "SEQUENTIAL_FALLBACK",
  "fallback_events": [],
  "rows_selected": 154,
  "questions_selected": [
    "Q01",
    "Q02",
    "Q03",
    "Q04",
    "Q05",
    "Q06",
    "Q07",
    "Q08",
    "Q09",
    "Q10",
    "Q11",
    "Q12",
    "Q13",
    "Q14"
  ],
  "coder_a_output_path": "/content/HFG/data/processed/coder_a_outputs.json",
  "coder_b_output_path": "/content/HFG/data/processed/coder_b_outputs.json",

AttributeError: 'PipelineEvaluationReport' object has no attribute 'to_dict'

In [10]:
import json

print(
    json.dumps(
        final_report.model_dump(mode="json"),
        indent=2,
        default=str,
    )
)

print(
    "\n================================"
)

print(
    "AGENT_03_RELEASE =",
    final_report.agent_03_release_decision.value,
)

print(
    "================================"
)

{
  "run_id": "run_20261111_002",
  "evaluated_at_utc": "2026-11-14T10:51:30.729798+00:00",
  "states": {
    "INGESTION_READY": true,
    "QUALITATIVE_CONTRACT_READY": true,
    "GPU_PREFLIGHT_READY": true,
    "CODER_A_COMPLETE": true,
    "CODER_B_COMPLETE": true,
    "DUAL_CODER_VALIDATED": true,
    "AGENT_03_RELEASE_READY": true
  },
  "highest_verified_state": "AGENT_03_RELEASE_READY",
  "agent_01_gate_1": "PASS",
  "agent_02_entrance_gate": "PASS",
  "agent_07_persisted": true,
  "live_gpu_hardware_preflight": "PASS",
  "live_gpu_runtime_benchmark": "PASS",
  "coder_a": {
    "coder_id": "coder_a",
    "execution_status": "PASS",
    "execution_profile": "production",
    "production_eligible": true,
    "synthetic_or_test_execution": false,
    "model_family": "deepseek",
    "base_model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
    "model_repo": "Valdemardi/DeepSeek-R1-Distill-Qwen-32B-AWQ",
    "model_revision": "1de6a3f7b151f6ea0f6d42acb3566e094eb8a264",
    "quantizati

In [11]:
print("Highest verified state:", final_report.highest_verified_state)
print("Blocking reasons:", final_report.blocking_reasons)

print("\nAgent 03 release checklist:")
for key, value in final_report.agent_03_release_checklist.items():
    print(f"{key}: {value}")

Highest verified state: AGENT_03_RELEASE_READY
Blocking reasons: []

Agent 03 release checklist:
1_agent_01_real_ingestion_pass: True
2_agent_02_entrance_contracts_pass: True
3_agent_07_persisted_in_git: True
4_live_gpu_benchmark_executed: True
5_coder_a_executed: True
6_coder_b_executed_independently: True
7_real_coder_a_output_exists: True
8_real_coder_b_output_exists: True
9_dual_outputs_schema_valid: True
10_dual_outputs_provenance_valid: True
11_missingness_no_synthetic_preference: True
12_agent_11_orchestration_and_freshness_pass: True
13_hardcoding_gate_pass: True


In [12]:
print(final_report.model_dump().keys())

dict_keys(['run_id', 'evaluated_at_utc', 'states', 'highest_verified_state', 'agent_01_gate_1', 'agent_02_entrance_gate', 'agent_07_persisted', 'live_gpu_hardware_preflight', 'live_gpu_runtime_benchmark', 'coder_a', 'coder_b', 'dual_coder_validation', 'hardcoding_scan_passed', 'freshness_checks_passed', 'agent_03_release_checklist', 'agent_03_release_decision', 'blocking_reasons'])


In [13]:
print("PRODUCTION STATUS =", prod_res.get("status"))

print("\nCoder A:")
print(
    json.dumps(
        prod_res.get("coder_a_metadata", {}),
        indent=2,
    )
)

print("\nCoder B:")
print(
    json.dumps(
        prod_res.get("coder_b_metadata", {}),
        indent=2,
    )
)

print(
    "\nValidation:",
    json.dumps(
        prod_res.get("validation_checks", {}),
        indent=2,
    )
)

print(
    "\nCheckpoint WAL:",
    prod_res.get("checkpoint_wal_path")
)

print(
    "Total checkpoints:",
    prod_res.get("total_checkpoints_persisted")
)

PRODUCTION STATUS = PASS

Coder A:
{
  "coder_id": "coder_a",
  "execution_profile": "production",
  "production_eligible": true,
  "synthetic_or_test_execution": false,
  "profile_config_hash": "",
  "model_family": "deepseek",
  "base_model": "deepseek-ai/DeepSeek-R1-Distill-Qwen-32B",
  "model_repo": "Valdemardi/DeepSeek-R1-Distill-Qwen-32B-AWQ",
  "model_revision": "1de6a3f7b151f6ea0f6d42acb3566e094eb8a264",
  "quantization": "AWQ-INT4",
  "serving_engine": "vllm_openai_server",
  "execution_mode": "SEQUENTIAL_FALLBACK",
  "server_backend": "vllm_openai_server",
  "model_load_succeeded": true,
  "gpu_resident": true,
  "inference_occurred": true,
  "responses_attempted": 136,
  "responses_completed": 136,
  "responses_failed": 0,
  "non_preference_cells_preserved": 18,
  "checkpoint_hits": 136,
  "prompt_hash": "4d1279f046aa4460707cb81b4e291cce30179cc060659fcdb4f796e224864377",
  "source_evidence_hash": "0b814c3b6f6a88ec8e5d895a025cb146ed2e73e08f814ace145c0237c17c9753",
  "reconcil

In [15]:
from pathlib import Path
import shutil
import hashlib

src = Path("/content/HFG/data/processed")
dst = Path(
    "/content/drive/MyDrive/HFG/run_20261111_002/"
    "production_artifacts"
)

dst.mkdir(parents=True, exist_ok=True)

for name in [
    "coder_a_outputs.json",
    "coder_b_outputs.json",
]:

    s = src / name
    d = dst / name

    assert s.exists(), s

    shutil.copy2(s, d)

    digest = hashlib.sha256(
        d.read_bytes()
    ).hexdigest()

    print(
        name,
        d.stat().st_size,
        digest,
    )

coder_a_outputs.json 565891 8939adc884672a1cdf883e496f476a872248d90d4f8da8f3549d29e24b20dadf
coder_b_outputs.json 706622 129c7257c1c4fdf9f3b88067b96180cf68ebd3056f41e0880fd5831d48ffcd49
